# PS3 Simulator — Stage 2: Sim-to-Real Evaluation
## MaCVi Workshop @ CVPR 2026

**Train: Synthetic PS3 only | Test: Real KSLG/SCTD**

Run cells in order: Cell 1 → Cell 12

In [ ]:
import os, gc, copy, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, datasets
from sklearn.metrics import (f1_score, classification_report,
                              confusion_matrix)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
CFG = {
    # ── Paths ──────────────────────────────────────────────────
    'data_dir'      : r'D:\phd\ijepa_data',
    'output_dir'    : r'D:\phd\ijepa_results_final',
    'ijepa_ckpt'    : r'D:\phd\ijepa_pretrain\logs\vits16-sonar\jepa-ep200.pth.tar',
    'ijepa_h_ckpt'  : r'C:\Users\T1_Machine\Downloads\IN1K-vit.h.14-300e.pth.tar',
    'dino_ps3_ckpt' : r'D:\phd\dino_pretrain\logs\checkpoint.pth',
    # ── Data ───────────────────────────────────────────────────
    'img_size'      : 224,
    'num_classes'   : 2,
    'batch_size'    : 32,
    # ── Training (Linear Probe — Random Init, ImageNet methods) 
    'lp_lr'         : 1e-3,
    'lp_epochs'     : 50,
    'lp_patience'   : 10,
    # ── Training (Full Fine-tune — PS3 methods) ────────────────
    'ft_lr'         : 1e-4,
    'ft_epochs'     : 100,
    'ft_patience'   : 20,
    # ── Common ─────────────────────────────────────────────────
    'weight_decay'  : 1e-4,
    'seeds'         : [42, 123, 456],
}

os.makedirs(CFG['output_dir'], exist_ok=True)

# Verify paths
print('\nPath check:')
print(f'  data_dir     : {os.path.exists(CFG["data_dir"])}')
print(f'  ijepa_ckpt   : {os.path.exists(CFG["ijepa_ckpt"])}')
print(f'  ijepa_h_ckpt : {os.path.exists(CFG["ijepa_h_ckpt"])}')
print(f'  dino_ps3_ckpt: {os.path.exists(CFG["dino_ps3_ckpt"])}')
print()
for split in ['train', 'val', 'test']:
    total = sum(
        len(os.listdir(os.path.join(CFG['data_dir'], split, cls)))
        for cls in ['ship', 'plane']
        if os.path.exists(os.path.join(CFG['data_dir'], split, cls))
    )
    tag = '  <- REAL sonar' if split == 'test' else ''
    print(f'  {split:5s}: {total} images{tag}')

In [ ]:
def get_transforms(split):
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(CFG['img_size']),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(45),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                  [0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((CFG['img_size'], CFG['img_size'])),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                  [0.229, 0.224, 0.225]),
        ])


def get_loaders():
    train_ds = datasets.ImageFolder(
        os.path.join(CFG['data_dir'], 'train'),
        get_transforms('train'))
    val_ds   = datasets.ImageFolder(
        os.path.join(CFG['data_dir'], 'val'),
        get_transforms('val'))
    test_ds  = datasets.ImageFolder(
        os.path.join(CFG['data_dir'], 'test'),
        get_transforms('test'))

    # Weighted sampler — handles class imbalance
    targets  = [s[1] for s in train_ds.samples]
    counts   = np.bincount(targets)
    weights  = torch.FloatTensor([1.0 / counts[t] for t in targets])
    sampler  = WeightedRandomSampler(weights, len(weights))

    train_loader = DataLoader(train_ds,
                              batch_size=CFG['batch_size'],
                              sampler=sampler,
                              num_workers=2,
                              pin_memory=True)
    val_loader   = DataLoader(val_ds,
                              batch_size=CFG['batch_size'],
                              shuffle=False,
                              num_workers=2,
                              pin_memory=True)
    test_loader  = DataLoader(test_ds,
                              batch_size=CFG['batch_size'],
                              shuffle=False,
                              num_workers=2,
                              pin_memory=True)

    print(f'Classes : {train_ds.classes}')
    print(f'Train   : {len(train_ds)} | '
          f'Val: {len(val_ds)} | '
          f'Test: {len(test_ds)}')
    return train_loader, val_loader, test_loader, train_ds.classes


train_loader, val_loader, test_loader, class_names = get_loaders()

In [ ]:
class LinearProbe(nn.Module):
    """Frozen backbone + trainable classification head."""
    def __init__(self, backbone, feat_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)


class FineTuneModel(nn.Module):
    """Full fine-tune backbone + trainable classification head."""
    def __init__(self, backbone, feat_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)


def freeze_backbone(model):
    """Freeze all backbone params, keep head trainable."""
    for name, param in model.named_parameters():
        if name.startswith('backbone.'):
            param.requires_grad = False
    return model


def get_feat_dim(backbone):
    backbone.eval()
    with torch.no_grad():
        dummy = torch.zeros(
            1, 3, CFG['img_size'], CFG['img_size']
        ).to(next(backbone.parameters()).device)
        out = backbone(dummy)
    return out.shape[-1]


def fix_pos_embed(sd, backbone):
    """Fix I-JEPA pos_embed shape mismatch [1,196,D]->[1,197,D]."""
    if 'pos_embed' in sd:
        pe_ckpt  = sd['pos_embed']
        pe_model = backbone.pos_embed
        if pe_ckpt.shape != pe_model.shape:
            cls_pe = pe_model[:, :1, :]
            sd['pos_embed'] = torch.cat([cls_pe, pe_ckpt], dim=1)
            print(f'  pos_embed fixed: '
                  f'{pe_ckpt.shape} -> {sd["pos_embed"].shape}')
    return sd


print('Model classes defined.')

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out   = model(imgs)
            loss  = criterion(out, labels)
            preds = out.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = correct / total
    f1  = f1_score(labels_all, preds_all, average='weighted')
    return total_loss / total, acc, f1, preds_all, labels_all


def run_experiment(model, method_name, lr, epochs,
                   patience, train_loader_override=None):
    tr_loader = (train_loader_override
                 if train_loader_override is not None
                 else train_loader)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs)

    best_f1, best_weights, patience_cnt = 0, None, 0
    history = {'tr_acc': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(
            model, tr_loader, criterion, optimizer)
        vl_loss, vl_acc, vl_f1, _, _ = evaluate(
            model, val_loader, criterion)
        scheduler.step()

        history['tr_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        history['val_f1'].append(vl_f1)

        if epoch % 10 == 0:
            print(f'    Ep{epoch:03d} '
                  f'tr={tr_acc:.3f} '
                  f'val={vl_acc:.3f} '
                  f'f1={vl_f1:.3f}')

        if vl_f1 > best_f1:
            best_f1      = vl_f1
            best_weights = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'    Early stop ep{epoch}')
                break

    model.load_state_dict(best_weights)
    _, test_acc, test_f1, preds, labels = evaluate(
        model, test_loader, criterion)
    print(f'    => TEST acc={test_acc*100:.1f}% '
          f'f1={test_f1:.4f}  [REAL sonar]')
    return {'accuracy': test_acc, 'f1': test_f1,
            'preds': preds, 'labels': labels,
            'history': history}


def run_3seeds(build_fn, method_name, lr, epochs, patience):
    """Run experiment 3 times with different seeds, return mean±std."""
    accs, f1s, all_preds, all_labels = [], [], [], []

    print(f'\n{"="*55}')
    print(f'Method : {method_name}')
    print(f'lr={lr} | epochs={epochs} | patience={patience}')
    print(f'{"="*55}')

    for seed in CFG['seeds']:
        torch.manual_seed(seed)
        np.random.seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.empty_cache()

        print(f'\n  Seed {seed}:')
        model  = build_fn()
        result = run_experiment(
            model, method_name, lr, epochs, patience)
        accs.append(result['accuracy'])
        f1s.append(result['f1'])
        all_preds.append(result['preds'])
        all_labels.append(result['labels'])

        del model
        torch.cuda.empty_cache()
        gc.collect()

    mean_acc = float(np.mean(accs))
    std_acc  = float(np.std(accs))
    mean_f1  = float(np.mean(f1s))
    std_f1   = float(np.std(f1s))

    # Use median seed preds for confusion matrix
    median_idx = int(np.argsort(accs)[len(accs) // 2])

    print(f'\n  FINAL {method_name}: '
          f'{mean_acc*100:.1f}±{std_acc*100:.1f}%  '
          f'F1={mean_f1:.4f}±{std_f1:.4f}')

    return {
        'method'  : method_name,
        'accuracy': mean_acc,
        'std'     : std_acc,
        'f1'      : mean_f1,
        'f1_std'  : std_f1,
        'preds'   : all_preds[median_idx],
        'labels'  : all_labels[median_idx],
        'history' : result['history'],
    }


def save_results(results,
                 path=None):
    if path is None:
        path = os.path.join(
            CFG['output_dir'], 'results_final.json')
    serializable = [
        {k: v for k, v in r.items()
         if k not in ['preds', 'labels', 'history']}
        for r in results
    ]
    with open(path, 'w') as f:
        json.dump(serializable, f, indent=2)
    print(f'Results saved: {path}')


print('Training functions defined.')

In [ ]:
print('\n' + '=' * 65)
print(f'{"METHOD":<25} {"ACC (%)":>10} {"±STD":>8} {"F1":>8}')
print('=' * 65)
for r in results:
    print(f'{r["method"]:<25} '
          f'{r["accuracy"]*100:>9.1f}% '
          f'{r["std"]*100:>7.1f}% '
          f'{r["f1"]:>8.4f}')
print('=' * 65)
print('Train: Synthetic PS3  |  Test: Real KSLG/SCTD')

# Per-class report
print('\nPer-class Classification Report:')
for r in results:
    print(f'\n{"-"*50}')
    print(f'Method: {r["method"]}')
    print(f'{"-"*50}')
    print(classification_report(
        r['labels'], r['preds'],
        target_names=class_names))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
fig.suptitle(
    'Confusion Matrices — Real Sonar Test Set\n'
    'Train: Synthetic PS3  |  Test: Real KSLG/SCTD',
    fontsize=13, fontweight='bold')

for ax, r in zip(axes, results):
    cm = confusion_matrix(r['labels'], r['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                ax=ax, cbar=False)
    acc = r['accuracy'] * 100
    std = r['std'] * 100
    ax.set_title(f'{r["method"]}\n'
                 f'Acc={acc:.1f}±{std:.1f}%',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
save_path = os.path.join(CFG['output_dir'], 'confusion_matrices.png')
plt.savefig(save_path, dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {save_path}')

In [ ]:
methods_names = [r['method'].replace(' ', '\n')
                 for r in results]
accs  = [r['accuracy'] * 100 for r in results]
stds  = [r['std'] * 100 for r in results]
f1s   = [r['f1'] * 100 for r in results]
colors = ['#d9534f', '#f0ad4e',
          '#5bc0de', '#aec6cf',
          '#5cb85c', '#2196F3']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Sim-to-Real Evaluation — Real Sonar Test Set\n'
    'Train: Synthetic PS3  |  Test: Real KSLG/SCTD',
    fontsize=13, fontweight='bold')

x = np.arange(len(methods_names))
w = 0.6

# Accuracy
axes[0].bar(x, accs, width=w,
            color=colors, yerr=stds,
            capsize=5, alpha=0.85,
            edgecolor='black', linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods_names, fontsize=8)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy ± Std')
axes[0].set_ylim(0, 100)
axes[0].grid(axis='y', alpha=0.3)
for i, (a, s) in enumerate(zip(accs, stds)):
    axes[0].text(i, a + s + 1, f'{a:.1f}',
                 ha='center', va='bottom',
                 fontsize=8, fontweight='bold')

# F1 Score
axes[1].bar(x, f1s, width=w,
            color=colors, alpha=0.85,
            edgecolor='black', linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods_names, fontsize=8)
axes[1].set_ylabel('F1 Score (%)')
axes[1].set_title('Weighted F1 Score')
axes[1].set_ylim(0, 100)
axes[1].grid(axis='y', alpha=0.3)
for i, f in enumerate(f1s):
    axes[1].text(i, f + 1, f'{f:.1f}',
                 ha='center', va='bottom',
                 fontsize=8, fontweight='bold')

plt.tight_layout()
save_path = os.path.join(CFG['output_dir'], 'bar_chart.png')
plt.savefig(save_path, dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {save_path}')

In [ ]:
def extract_features(backbone, loader, use_device='cpu'):
    """Extract features from backbone on test set."""
    backbone.eval()
    backbone = backbone.to(use_device)
    features, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs  = imgs.to(use_device)
            feats = backbone(imgs)
            features.append(feats.cpu().numpy())
            labels_all.append(labels.numpy())
    return (np.concatenate(features),
            np.concatenate(labels_all))


def plot_tsne(feats, labels, ax, title, class_names):
    """Run PCA then t-SNE and plot on ax."""
    n_components = min(50, feats.shape[1])
    feats_pca = PCA(
        n_components=n_components
    ).fit_transform(feats)
    f2d = TSNE(
        n_components=2,
        random_state=42,
        perplexity=30,
        n_iter=1000
    ).fit_transform(feats_pca)
    colors = ['#e74c3c', '#3498db']
    for cls_idx, (cls_name, color) in enumerate(
            zip(class_names, colors)):
        mask = labels == cls_idx
        ax.scatter(f2d[mask, 0], f2d[mask, 1],
                   c=color, label=cls_name,
                   alpha=0.6, s=15)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(fontsize=8)


print('Extracting features for t-SNE...')
torch.cuda.empty_cache()
gc.collect()

# Extract features for ViT-S methods (GPU)
bb_rand = timm.create_model(
    'vit_small_patch16_224',
    pretrained=False, num_classes=0)
feats_rand, lbl_rand = extract_features(
    bb_rand, test_loader, device)
print('  Random Init done')
del bb_rand; torch.cuda.empty_cache()

bb_dino_ps3 = timm.create_model(
    'vit_small_patch16_224',
    pretrained=False, num_classes=0)
ckpt = torch.load(
    CFG['dino_ps3_ckpt'],
    map_location='cpu', weights_only=False)
sd = {k.replace('module.', '').replace('backbone.', ''): v
      for k, v in ckpt['student'].items()}
bb_dino_ps3.load_state_dict(sd, strict=False)
feats_dino_ps3, lbl_dino_ps3 = extract_features(
    bb_dino_ps3, test_loader, device)
print('  DINO (PS3) done')
del bb_dino_ps3; torch.cuda.empty_cache()

bb_ijepa_ps3 = timm.create_model(
    'vit_small_patch16_224',
    pretrained=False, num_classes=0)
ckpt = torch.load(
    CFG['ijepa_ckpt'],
    map_location='cpu', weights_only=False)
if 'target_encoder' in ckpt:
    sd = {k.replace('module.', ''): v
          for k, v in ckpt['target_encoder'].items()}
else:
    sd = {k.replace('module.', ''): v
          for k, v in ckpt['encoder'].items()}
sd = fix_pos_embed(sd, bb_ijepa_ps3)
bb_ijepa_ps3.load_state_dict(sd, strict=False)
feats_ijepa_ps3, lbl_ijepa_ps3 = extract_features(
    bb_ijepa_ps3, test_loader, device)
print('  I-JEPA (PS3) done')
del bb_ijepa_ps3; torch.cuda.empty_cache()

bb_dino_in = timm.create_model(
    'vit_small_patch16_224_dino',
    pretrained=True, num_classes=0)
feats_dino_in, lbl_dino_in = extract_features(
    bb_dino_in, test_loader, device)
print('  DINO (ImageNet) done')
del bb_dino_in; torch.cuda.empty_cache()

# ViT-H on CPU to avoid OOM
bb_ijepa_h = timm.create_model(
    'vit_huge_patch14_224',
    pretrained=False, num_classes=0)
ckpt = torch.load(
    CFG['ijepa_h_ckpt'],
    map_location='cpu', weights_only=False)
if 'target_encoder' in ckpt:
    sd = {k.replace('module.', ''): v
          for k, v in ckpt['target_encoder'].items()}
else:
    sd = {k.replace('module.', ''): v
          for k, v in ckpt['encoder'].items()}
sd = fix_pos_embed(sd, bb_ijepa_h)
bb_ijepa_h.load_state_dict(sd, strict=False)
feats_ijepa_h, lbl_ijepa_h = extract_features(
    bb_ijepa_h, test_loader, 'cpu')  # CPU for ViT-H
print('  I-JEPA (ImageNet) ViT-H done')
del bb_ijepa_h; gc.collect()

# Plot t-SNE
acc_map = {r['method']: r['accuracy'] * 100
           for r in results}
tsne_data = [
    ('Random Init\n'
     f'({acc_map["Random Init"]:.1f}%)',
     feats_rand, lbl_rand),
    ('DINO (PS3)\n'
     f'({acc_map["DINO (PS3)"]:.1f}%)',
     feats_dino_ps3, lbl_dino_ps3),
    ('I-JEPA PS3 (Ours)\n'
     f'({acc_map["I-JEPA (PS3)"]:.1f}%)',
     feats_ijepa_ps3, lbl_ijepa_ps3),
    ('DINO (ImageNet)\n'
     f'({acc_map["DINO (ImageNet)"]:.1f}%)',
     feats_dino_in, lbl_dino_in),
    ('I-JEPA ImageNet ViT-H\n'
     f'({acc_map["I-JEPA (ImageNet)"]:.1f}%)',
     feats_ijepa_h, lbl_ijepa_h),
]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle(
    't-SNE Feature Visualisation — Real Sonar Test Set\n'
    'Train: Synthetic PS3  |  Test: Real KSLG/SCTD',
    fontsize=13, fontweight='bold')

for ax, (title, feats, labels) in zip(axes, tsne_data):
    plot_tsne(feats, labels, ax, title, class_names)

plt.tight_layout()
save_path = os.path.join(CFG['output_dir'], 'tsne_final.png')
plt.savefig(save_path, dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {save_path}')